# 02 BUT PPG Preprocessing

This notebook demonstrates the exact preprocessing contract used by the
offline dataset builder and future FastAPI inference service.

In [ ]:
from pathlib import Path
import os
import sys


def find_repository_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / ".git").exists() and (candidate / "apps").exists():
            return candidate
    raise RuntimeError("Run this notebook from inside the pure-pulsesure repository")


REPO_ROOT = find_repository_root(Path.cwd().resolve())
AI_SERVICE_ROOT = REPO_ROOT / "apps" / "ai-service"
sys.path.insert(0, str(AI_SERVICE_ROOT))

RAW_DIR = Path(
    os.environ.get("PURE_BUT_PPG_DIR", REPO_ROOT / "data" / "raw" / "but-ppg-2.0.0")
).expanduser().resolve()
PROCESSED_DIR = Path(
    os.environ.get("PURE_PROCESSED_DIR", REPO_ROOT / "data" / "processed" / "but-ppg-v2")
).expanduser().resolve()
MODEL_DIR = Path(os.environ.get("PURE_MODEL_DIR", REPO_ROOT / "models")).expanduser().resolve()

print("Repository:", REPO_ROOT)
print("Raw dataset:", RAW_DIR)
print("Processed dataset:", PROCESSED_DIR)
print("Model directory:", MODEL_DIR)

In [ ]:
import json
import subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from app.services.preprocessing import (
    PreprocessingConfig,
    extract_signal_features,
    load_but_ppg_record,
    preprocess_ppg,
)

CONFIG = PreprocessingConfig()
CONFIG.to_dict()

In [ ]:
record_ids = ["100001", "146014"]
rows = []
fig, axes = plt.subplots(len(record_ids), 2, figsize=(14, 6), sharex=True)

for row_index, record_id in enumerate(record_ids):
    record = load_but_ppg_record(RAW_DIR / record_id)
    processed = preprocess_ppg(record.waveform, CONFIG)
    features = extract_signal_features(record.waveform, processed, record.sampling_rate_hz)
    rows.append({
        "record_id": record_id,
        "layout": record.source_layout,
        "channel": record.selected_channel,
        **features,
    })
    time = np.arange(record.waveform.size) / record.sampling_rate_hz
    axes[row_index, 0].plot(time, record.waveform, color="#142638")
    axes[row_index, 0].set_title(f"{record_id} raw | {record.source_layout}")
    axes[row_index, 1].plot(time, processed, color="#35C4B5")
    axes[row_index, 1].set_title(f"{record_id} filtered and normalized")

plt.tight_layout()
display(pd.DataFrame(rows).T)

In [ ]:
# First validate the complete path on a small subset.
smoke_output = REPO_ROOT / "data" / "processed" / "but-ppg-notebook-smoke"
command = [
    sys.executable,
    str(REPO_ROOT / "scripts" / "prepare_but_ppg.py"),
    "--raw-dir", str(RAW_DIR),
    "--output-dir", str(smoke_output),
    "--limit", "100",
]
subprocess.run(command, check=True)

In [ ]:
smoke_manifest = json.loads((smoke_output / "manifest.json").read_text())
display(pd.DataFrame(smoke_manifest["split_summary"]).T)
print("Failures:", smoke_manifest["records_failed"])

In [ ]:
# Change to True only after the smoke run succeeds.
RUN_FULL_PREPROCESSING = False

if RUN_FULL_PREPROCESSING:
    full_command = [
        sys.executable,
        str(REPO_ROOT / "scripts" / "prepare_but_ppg.py"),
        "--raw-dir", str(RAW_DIR),
        "--output-dir", str(PROCESSED_DIR),
    ]
    subprocess.run(full_command, check=True)
else:
    print("Full preprocessing skipped. Set RUN_FULL_PREPROCESSING=True to run it.")

In [ ]:
if (PROCESSED_DIR / "manifest.json").exists():
    manifest = json.loads((PROCESSED_DIR / "manifest.json").read_text())
    display(pd.DataFrame(manifest["split_summary"]).T)
    print("Processed:", manifest["records_processed"], "Failed:", manifest["records_failed"])
else:
    print("Run the full preprocessing cell before training.")